<a href="https://colab.research.google.com/github/AnkitByteForge/llm-orchestrator-router-memory/blob/main/Routing_Driven_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install dependencies

In [ ]:
!pip -q install openai

## Get API key from Colab Secrets

In [ ]:
# Colab-specific utility to access Secrets
from google.colab import userdata
import os

# Load OpenAI API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI")

# Choose the model for the entire notebook
# Keeping this as an env var makes switching models trivial
os.environ["OPENAI_MODEL"] = "gpt-4o-mini"

# We print only whether the key exists, never the key itself
print("Key set:", bool(os.environ.get("OPENAI_API_KEY")))
print("Model:", os.environ["OPENAI_MODEL"])

Key set: True
Model: gpt-4o-mini


## Imports + Settings

In [ ]:
from __future__ import annotations

import os
import json
import re
import sqlite3
from dataclasses import dataclass
from typing import List, Literal, Dict, Any
from datetime import datetime, timezone

from openai import OpenAI

# Route is a bounded set of allowed routing outcomes.
# This is CRITICAL for safe routing:
# - The router is NOT allowed to invent new handlers
# - Every route must map to a known execution path
Route = Literal["BILLING", "TECH", "GENERAL"]

# Settings object
# - Holds configuration for the entire system
@dataclass(frozen=True)
class Settings:
    model: str
    sqlite_path: str

# Load settings from environment variables
# - OPENAI_MODEL controls which LLM is used everywhere
# - sqlite_path points to the local SQLite file used for memory
# - Using env vars makes this portable across Colab / local / prod
def load_settings() -> Settings:
    return Settings(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        # SQLite DB file lives inside the Colab runtime filesystem
        # This persists across notebook cells but resets on runtime restart
        sqlite_path="memory.db",
    )

settings = load_settings()
client = OpenAI()

# Print settings so learners can SEE what configuration is active
settings

Settings(model='gpt-4o-mini', sqlite_path='memory.db')

## OpenAI helper (Responses API) + JSON extraction

In [ ]:
def _extract_json(text: str) -> Dict[str, Any]:
    """
        Even when you explicitly instruct an LLM to return ONLY JSON,
        models can still:
        - prepend explanations
        - append comments
        - wrap JSON in markdown
        - add stray newlines or text
    This function defensively:
    1) checks for clean JSON
    2) otherwise extracts the first {...} block it can find
    """
    s = str(text).strip()
    # If the entire response is already a valid JSON object, parse directly.
    # This is the ideal and most common case when prompts are followed correctly.
    if s.startswith("{") and s.endswith("}"):
        return json.loads(s)

    # If the model added extra text, search for the FIRST JSON object anywhere
    # in the response using a regex.
    #
    # DOTALL allows '.' to match newlines so multi-line JSON is captured.
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)

    # If we still can't find JSON, this is a HARD FAILURE.
    # We raise immediately instead of guessing.
    if not m:
        raise ValueError(f"Expected JSON object. Got: {s[:400]}")
    return json.loads(m.group(0))

def openai_response_text(client: OpenAI, model: str, messages: List[Dict[str, Any]]) -> str:
    """
    Uses OpenAI Responses API.
    messages: [{"role": "system"|"user", "content": "..."}]
    """
    resp = client.responses.create(
        model=model,
        input=messages,
    )
    # Prefer output_text if available; fallback to str
    return getattr(resp, "output_text", None) or str(resp)

## Router (OpenAI) [Section 1: Routing Decisions]


In [ ]:
ROUTER_PROMPT = """
You are the Router in a routing-driven system.

Return ONLY valid JSON (no markdown, no extra text).
Schema:
{
  "route": "BILLING" | "TECH" | "GENERAL",
  "reason": "<short reason, <= 20 words>",
  "questions": ["<q1>", "<q2>", "<q3>"]
}

Rules:
- BILLING: refunds, invoices, GST, payments, subscriptions, pricing, card issues.
- TECH: errors, bugs, login/OTP, integrations, APIs, timeouts, performance.
- GENERAL: unclear or mixed; ask MINIMUM questions to decide BILLING vs TECH.
Constraints:
- questions length <= 3
"""

@dataclass(frozen=True)
class RouteDecision:
    route: Route
    reason: str
    questions: List[str]

def route_with_openai(client: OpenAI, model: str, user_text: str, memory_context: str) -> RouteDecision:
    """
    This function is the DECISION LAYER of the system.

    It answers exactly one question:
        "Who should handle this request?"

    It does NOT:
    - solve the user's problem
    - generate final answers
    - perform any execution

    Flow:
    1) Prepare routing inputs (user request + short memory)
    2) Ask the LLM to make a bounded routing decision
    3) Parse and validate the output defensively
    4) Normalize and constrain the decision
    5) Return an immutable RouteDecision object
    """
    raw = openai_response_text(
        client,
        model,
        [
            {"role": "system", "content": ROUTER_PROMPT},
            {"role": "system", "content": f"Memory context:\n{memory_context}"},
            {"role": "user", "content": user_text},
        ],
    )

    data = _extract_json(raw)

    route = str(data.get("route", "GENERAL")).strip().upper()
    if route not in ("BILLING", "TECH", "GENERAL"):
        route = "GENERAL"

    reason = (str(data.get("reason", "")) or "").strip() or "No reason provided."
    questions = data.get("questions") or []
    questions = [str(q).strip() for q in questions if str(q).strip()][:3]

    return RouteDecision(route=route, reason=reason, questions=questions)

## Tasks (Functions) + Roles (Prompts) [Section 2]

Tasks are equivalent to Python functions.
Roles are prompt templates.

In [ ]:
BILLING_ROLE_PROMPT = """
ROLE: Billing Specialist
SCOPE: refunds, invoices, GST, subscription changes, pricing, payments, card updates.
OUT OF SCOPE: debugging product errors, integrations, OTP delivery troubleshooting.

OUTPUT FORMAT:
1) Summary
2) Resolution Steps (numbered)
3) Required Info (only if needed)
4) Next Actions
"""

TECH_ROLE_PROMPT = """
ROLE: Tech Support Specialist
SCOPE: errors, bugs, login/OTP, integrations, APIs, timeouts, performance.
OUT OF SCOPE: refunds/invoices/pricing decisions.

OUTPUT FORMAT:
1) Summary
2) Diagnostics Needed (if any)
3) Troubleshooting Steps (numbered)
4) Next Actions
"""

CLARIFIER_ROLE_PROMPT = """
ROLE: Clarifier
You do NOT solve the issue.

If it a general chat, just continue it

Ask up to 3 minimal questions to decide BILLING vs TECH (ONLY IF USER is requesting for some issues)
OUTPUT:
1) <question>
2) <question>
3) <question>
"""

def billing_task(client: OpenAI, model: str, user_text: str, memory_context: str, decision: RouteDecision) -> str:
    return openai_response_text(
        client, model,
        [
            {"role": "system", "content": BILLING_ROLE_PROMPT},
            {"role": "system", "content": f"Memory context:\n{memory_context}"},
            {"role": "system", "content": f"Routing: {decision.route} | Reason: {decision.reason}"},
            {"role": "user", "content": user_text},
        ],
    )

def tech_task(client: OpenAI, model: str, user_text: str, memory_context: str, decision: RouteDecision) -> str:
    return openai_response_text(
        client, model,
        [
            {"role": "system", "content": TECH_ROLE_PROMPT},
            {"role": "system", "content": f"Memory context:\n{memory_context}"},
            {"role": "system", "content": f"Routing: {decision.route} | Reason: {decision.reason}"},
            {"role": "user", "content": user_text},
        ],
    )

def clarifier_task(client: OpenAI, model: str, user_text: str, memory_context: str, decision: RouteDecision) -> str:
    hints = decision.questions or []
    return openai_response_text(
        client, model,
        [
            {"role": "system", "content": CLARIFIER_ROLE_PROMPT},
            {"role": "system", "content": f"Memory context:\n{memory_context}"},
            {"role": "system", "content": f"Suggested questions (use only if needed): {hints}"},
            {"role": "user", "content": user_text},
        ],
    )

## SQLite Memory Store (schema + helpers)

- store turns (user_id, ts, role, content)
- fetch last N turns
- compose context string
- clear user memory

In [ ]:
# MEMORY LAYER — SQLite-backed short-term memory
#
# PURPOSE:
# - Persist conversation turns across requests
# - Enable continuity without relying on model hallucination
# - Provide a small, bounded context to routing + execution layers
#
# DESIGN CHOICES:
# - SQLite (simple, local, deterministic)
# - One table, append-only
# - Memory = last N turns, not full chat logs

def now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sqlite_init(db_path: str) -> sqlite3.Connection:
    """
    Initialize the SQLite database and memory table.

    This function is responsible for:
    - Creating the database file (if it doesn't exist)
    - Creating the memory table
    - Creating indexes for fast retrieval

    IMPORTANT:
    - This is infrastructure code, not agent logic
    - Memory should exist independently of routing/execution
    """
    conn = sqlite3.connect(db_path)
    # conversation_turns table
    #
    # Each row represents ONE turn:
    # - user message OR assistant output
    #
    # Columns:
    # - id       : surrogate primary key
    # - user_id  : identity boundary (VERY important)
    # - ts       : timestamp for ordering
    # - role     : 'user' or 'assistant'
    # - content  : raw text content
    #
    # We explicitly constrain role to ('user', 'assistant')
    # to avoid corrupt or ambiguous memory entries.
    conn.execute("""
    CREATE TABLE IF NOT EXISTS conversation_turns (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT NOT NULL,
        ts TEXT NOT NULL,
        role TEXT NOT NULL CHECK(role IN ('user','assistant')),
        content TEXT NOT NULL
    )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_user_ts ON conversation_turns(user_id, ts)")
    conn.commit()
    return conn

def memory_append_turn(conn: sqlite3.Connection, user_id: str, role: Literal["user","assistant"], content: str) -> None:
    """
    Append a single conversation turn to memory.

    Key idea:
    - Memory is WRITE-ONCE, APPEND-ONLY
    - We NEVER mutate past memory

    Why this matters:
    - Deterministic debugging
    - Clear audit trail
    - No hidden state mutation

    We store:
    - user input
    - assistant output
    as separate rows.
    """
    conn.execute(
        "INSERT INTO conversation_turns(user_id, ts, role, content) VALUES (?,?,?,?)",
        (user_id, now_iso(), role, content),
    )
    conn.commit()

def memory_fetch_last_turns(conn: sqlite3.Connection, user_id: str, limit: int = 10) -> List[Dict[str, Any]]:
    """
    Fetch the last N conversation turns for a user.
    """
    rows = conn.execute(
        "SELECT user_id, ts, role, content FROM conversation_turns WHERE user_id=? ORDER BY ts DESC LIMIT ?",
        (user_id, limit),
    ).fetchall()
    rows = list(reversed(rows))  # oldest -> newest
    return [{"user_id": r[0], "ts": r[1], "role": r[2], "content": r[3]} for r in rows]

def memory_fetch_context(conn: sqlite3.Connection, user_id: str, limit: int = 5) -> str:
    """
    Build a SHORT, BOUNDED memory context string.

    This is the memory that is actually passed to the LLM.
    """
    turns = memory_fetch_last_turns(conn, user_id=user_id, limit=limit)
    if not turns:
        return "No prior memory found for this user."

    lines = []
    for t in turns:
        role = t["role"].upper()
        content = (t["content"] or "").strip()
        if len(content) > 500:
            content = content[:500] + "…"
        lines.append(f"{role}: {content}")
    return "Recent conversation (last turns):\n" + "\n".join(lines)

def memory_clear_user(conn: sqlite3.Connection, user_id: str) -> int:
    cur = conn.execute("DELETE FROM conversation_turns WHERE user_id=?", (user_id,))
    conn.commit()
    return cur.rowcount

# Init DB
conn = sqlite_init(settings.sqlite_path)
print("SQLite initialized at:", settings.sqlite_path)

SQLite initialized at: memory.db


## Orchestrator (if/else) + Memory Writeback [Section 3 + Integration]

This is the core loop:
- read SQLite memory → build context (last 5 turns)
- call router (OpenAI)
- if/else to execute the correct task function
- store user + assistant output back into SQLite

In [ ]:
# @title
def run_system(conn: sqlite3.Connection, client: OpenAI, settings: Settings, user_id: str, user_text: str) -> Dict[str, Any]:
    # 1) read memory
    # memory_context = memory_fetch_context(conn, user_id=user_id, limit=5)
    memory_context = None # for testing memory demo

    # 2) route
    decision = route_with_openai(client, settings.model, user_text, memory_context)

    # 3) execute via if/else
    if decision.route == "BILLING":
        output = billing_task(client, settings.model, user_text, memory_context, decision)
    elif decision.route == "TECH":
        output = tech_task(client, settings.model, user_text, memory_context, decision)
    else:
        output = clarifier_task(client, settings.model, user_text, memory_context, decision)

    # 4) write memory
    memory_append_turn(conn, user_id, "user", user_text)
    memory_append_turn(conn, user_id, "assistant", output)

    return {
        "route": decision.route,
        "reason": decision.reason,
        "questions": decision.questions,
        "memory_context": memory_context,
        "output": output,
    }

## Example Runs

In [ ]:
user_id = "test-user-1"

examples = [
    "Hi My Name is Chirag!",
    "Do you remember my name, I want to raise a complaint request with my system shutting down on loop!",
]

for text in examples:
    print("\n" + "="*90)
    print("USER:", text)
    result = run_system(conn, client, settings, user_id, text)
    print("\nROUTER:", {"route": result["route"], "reason": result["reason"], "questions": result["questions"]})
    print("\nMEMORY CONTEXT USED:\n", result["memory_context"])
    print("\nFINAL OUTPUT:\n", result["output"][:1200], "...")


USER: Hi My Name is Chirag!

ROUTER: {'route': 'GENERAL', 'reason': 'User introduction; no specific issue stated.', 'questions': ['What assistance do you need?', 'Is it related to billing or tech?', 'Can you describe the issue?']}

MEMORY CONTEXT USED:
 No prior memory found for this user.

FINAL OUTPUT:
 Hi Chirag! Nice to meet you! What’s on your mind today? ...

USER: Do you remember my name, I want to raise a complaint request with my system shutting down on loop!

ROUTER: {'route': 'TECH', 'reason': 'System is experiencing continuous shutdowns.', 'questions': ['When did this issue start?', 'Have you made any recent changes to your system?', 'What operating system are you using?']}

MEMORY CONTEXT USED:
 Recent conversation (last turns):
USER: Hi My Name is Chirag!
ASSISTANT: Hi Chirag! Nice to meet you! What’s on your mind today?

FINAL OUTPUT:
 1) **Summary**  
Your system is experiencing continuous shutdowns, which is creating a loop and preventing normal operation.

2) **Diagn

In [ ]:
user_id = "test-user-3"

examples = [
    "I was charged twice for my subscription. Invoice INV-20491. Email: abc@company.com.",
    "Any update on that? Use what you already know from my last message.",
]

for text in examples:
    print("\n" + "="*90)
    print("USER:", text)
    result = run_system(conn, client, settings, user_id, text)
    print("\nROUTER:", {"route": result["route"], "reason": result["reason"], "questions": result["questions"]})
    print("\nMEMORY CONTEXT USED:\n", result["memory_context"])
    print("\nFINAL OUTPUT:\n", result["output"][:1200], "...")


USER: I was charged twice for my subscription. Invoice INV-20491. Email: abc@company.com.

ROUTER: {'route': 'BILLING', 'reason': 'Duplicate charge for subscription', 'questions': []}

MEMORY CONTEXT USED:
 None

FINAL OUTPUT:
 1) **Summary**: The customer was charged twice for their subscription, as indicated by invoice INV-20491.

2) **Resolution Steps**:
   1. Verify the payment records for invoice INV-20491 to confirm the duplicate charge.
   2. Review the customer's subscription details to ensure no errors in subscription management.
   3. Process a refund for the duplicate charge if confirmed.
   4. Update the customer via email regarding the refund and any further steps if needed.

3) **Required Info**: None needed at this stage.

4) **Next Actions**: Execute the resolution steps and confirm the refund to the customer. Notify the customer at abc@company.com about the resolution and refund status. ...

USER: Any update on that? Use what you already know from my last message.

RO

## Inspect stored SQLite history

In [ ]:
turns = memory_fetch_last_turns(conn, user_id=user_id, limit=20)

print(f"Total fetched turns: {len(turns)}\n")
for t in turns:
    print(f"{t['ts']} | {t['role'].upper()}: {t['content'][:120]}{'…' if len(t['content'])>120 else ''}")

Total fetched turns: 4

2026-02-10T13:34:10+00:00 | USER: Hi My Name is Chirag!
2026-02-10T13:34:10+00:00 | ASSISTANT: Hi Chirag! Nice to meet you! What’s on your mind today?
2026-02-10T13:34:18+00:00 | USER: Do you remember my name, I want to raise a complaint request with my system shutting down on loop!
2026-02-10T13:34:18+00:00 | ASSISTANT: 1) **Summary**  
Your system is experiencing continuous shutdowns, which is creating a loop and preventing normal operat…
